[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/hierarchical_classification.ipynb)

# Classify down a tree, and stop when confidence drops

A flat list of every leaf is a long question, and a wrong leaf is worse than an honest coarser label. At each level FRIDA-Decisions chooses among the children of the nodes still in the beam. Code multiplies the probabilities down the path. If the step into a node is not confident, the label falls back to the parent.

`confidence` here is the confidence of the choice that entered the node, not a probability of the whole path. The path probability is the product, and the fallback uses confidence.

The tree and the products are synthetic.


In [ ]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [1]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


The tree, the beam, and the fallback. Each node stores only the confidence of the choice that entered it. A node the beam kept without choosing it has no confidence, so the label walks back to the parent.


In [2]:
DATA = [
    {"gold": "smartphones", "text": "Смартфон с сенсорным экраном, 128 ГБ, две сим-карты."},
    {"gold": "button", "text": "Кнопочный телефон с крупными кнопками и без сенсорного экрана."},
    {"gold": "headphones", "text": "Беспроводные наушники с шумоподавлением, кейс в комплекте."},
    {"gold": "kettles", "text": "Электрический чайник на 1,7 литра, металлическая колба."},
    {"gold": "chairs", "text": "Офисный стул на колёсиках, высота сиденья регулируется."},
    {"gold": "black", "text": "Пачка чёрного чая, сто пакетиков, без добавок."},
    {"gold": "speakers", "text": "Портативная колонка с блютузом, для музыки в комнате."},
    {"gold": "milk", "text": "Молоко 3,2 процента, один литр, в бутылке."},
]
TREE = {
    "goods": {"electronics": "электроника и гаджеты", "home": "товары для дома и кухни",
              "grocery": "продукты и напитки"},
    "electronics": {"phones": "телефоны", "audio": "наушники и колонки"},
    "home": {"kitchen": "техника для кухни", "furniture": "мебель"},
    "grocery": {"tea": "чай и кофе", "dairy": "молочные продукты"},
    "phones": {"smartphones": "смартфоны с сенсорным экраном", "button": "кнопочные телефоны"},
    "audio": {"headphones": "наушники", "speakers": "колонки"},
    "kitchen": {"kettles": "чайники и термопоты", "coffee": "кофеварки и кофемолки"},
    "furniture": {"chairs": "стулья", "tables": "столы"},
    "tea": {"black": "чёрный чай", "green": "зелёный чай"},
    "dairy": {"milk": "молоко", "cheese": "сыр"},
}
PARENT = {child: node for node, kids in TREE.items() for child in kids}
BEAM = 2
MIN_CONFIDENCE = 0.45


def classify(judge, text):
    frontier = [("goods", 1.0)]
    entered = {}
    while True:
        expandable = []
        for node, prob in frontier:
            if node in TREE:
                expandable.append((node, prob))
        if not expandable:
            break
        questions = {}
        for node, _prob in expandable:
            questions[node] = {
                "type": "choice",
                "instructions": "К какой категории из списка относится товар?",
                "criteria": TREE[node],
            }
        answers = judge({"state": text, "questions": questions})["answers"]
        nxt = []
        for node, prob in expandable:
            answer = answers[node]
            for child, child_p in answer["probabilities"].items():
                if child == answer["choice"]:
                    entered[child] = answer["confidence"]
                nxt.append((child, prob * child_p))
        nxt.sort(key=lambda item: -item[1])
        frontier = nxt[:BEAM]
    best, path_p = frontier[0]
    label = best
    while label != "goods" and entered.get(label, 0.0) < MIN_CONFIDENCE:
        label = PARENT[label]
    return {"leaf": best, "label": label, "path_p": path_p, "confidence": entered.get(best)}


def covers(label, gold):
    node = gold
    while node is not None:
        if node == label:
            return True
        node = PARENT.get(node)
    return False


`leaf` is the end of the best path before the fallback. `label` is what the code would store. A coarse hit counts when that label is the gold leaf or one of its ancestors.


In [3]:
def run(judge):
    exact = 0
    coarse = 0
    for row in DATA:
        result = classify(judge, row["text"])
        if result["label"] == row["gold"]:
            mark = "exact"
            exact += 1
            coarse += 1
        elif covers(result["label"], row["gold"]):
            mark = "coarse"
            coarse += 1
        else:
            mark = "miss"
        print(f"{mark:6} gold {row['gold']:12} label {result['label']:12} "
              f"leaf {result['leaf']:12} path_p={result['path_p']:.3f} "
              f"leaf_conf={result['confidence']:.2f}")
    print(f"\nexact {exact}/{len(DATA)}, gold or an ancestor of gold {coarse}/{len(DATA)}, "
          f"beam {BEAM}, confidence cutoff {MIN_CONFIDENCE}")


In [4]:
run(judge)


exact  gold smartphones  label smartphones  leaf smartphones  path_p=0.998 leaf_conf=1.00
exact  gold button       label button       leaf button       path_p=0.988 leaf_conf=0.94
exact  gold headphones   label headphones   leaf headphones   path_p=0.941 leaf_conf=0.98
exact  gold kettles      label kettles      leaf kettles      path_p=0.960 leaf_conf=1.00
exact  gold chairs       label chairs       leaf chairs       path_p=0.946 leaf_conf=0.79
exact  gold black        label black        leaf black        path_p=0.956 leaf_conf=1.00
exact  gold speakers     label speakers     leaf speakers     path_p=0.647 leaf_conf=0.67
exact  gold milk         label milk         leaf milk         path_p=0.998 leaf_conf=1.00

exact 8/8, gold or an ancestor of gold 8/8, beam 2, confidence cutoff 0.45
